In [ ]:
!pip install plotly ipywidgets pandas numpy

from google.colab import output
output.enable_custom_widget_manager()

import plotly.io as pio
pio.renderers.default = "colab"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.7 MB/s eta 0:00:00


In [3]:
# ============================================================
# DASHBOARD INTERACTIVO ECG - VERSION COLAB
# Simulación de señales cardíacas para docencia biomédica
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

np.random.seed(123)

# ============================================================
# UTILIDADES
# ============================================================

def gaussian(t, mu, sigma, amp):
    return amp * np.exp(-0.5 * ((t - mu) / sigma) ** 2)

def moving_average(x, w):
    w = max(1, int(w))
    return np.convolve(x, np.ones(w) / w, mode="same")

# ============================================================
# MODELO DE LATIDO ECG
# ============================================================

def single_beat_template(fs=1000, rr=1.0, amplitude=1.0, rhythm_type="Normal"):
    """
    Genera un latido ECG sintético con ondas P, QRS y T.
    """
    beat_len = int(rr * fs)
    t = np.linspace(0, rr, beat_len, endpoint=False)
    x = np.zeros_like(t)

    # Posiciones relativas
    p_pos = 0.18 * rr
    q_pos = 0.36 * rr
    r_pos = 0.40 * rr
    s_pos = 0.44 * rr
    t_pos = 0.68 * rr

    # Anchuras
    p_w = 0.035 * rr
    q_w = 0.010 * rr
    r_w = 0.012 * rr
    s_w = 0.012 * rr
    t_w = 0.080 * rr

    # Amplitudes base
    p_a = 0.12 * amplitude
    q_a = -0.15 * amplitude
    r_a = 1.00 * amplitude
    s_a = -0.25 * amplitude
    t_a = 0.35 * amplitude

    if rhythm_type == "Taquicardia":
        t_a = 0.28 * amplitude
        p_a = 0.10 * amplitude

    elif rhythm_type == "Bradicardia":
        t_a = 0.38 * amplitude

    elif rhythm_type == "Extrasistole":
        # Latido más prematuro con QRS algo diferente
        r_a = 1.15 * amplitude
        q_a = -0.20 * amplitude
        s_a = -0.35 * amplitude
        t_a = 0.22 * amplitude

    elif rhythm_type == "FA simulada":
        # Sin onda P organizada
        p_a = 0.0
        t_a = 0.30 * amplitude

    x += gaussian(t, p_pos, p_w, p_a)
    x += gaussian(t, q_pos, q_w, q_a)
    x += gaussian(t, r_pos, r_w, r_a)
    x += gaussian(t, s_pos, s_w, s_a)
    x += gaussian(t, t_pos, t_w, t_a)

    return t, x

# ============================================================
# GENERACION DE ECG COMPLETO
# ============================================================

def generate_rr_series(duration, hr, rhythm_type):
    """
    Devuelve los intervalos RR.
    """
    mean_rr = 60.0 / hr
    rr_list = []
    total = 0.0

    while total < duration + 1.0:
        if rhythm_type == "Normal":
            rr = np.random.normal(mean_rr, 0.03)

        elif rhythm_type == "Taquicardia":
            rr = np.random.normal(mean_rr, 0.02)

        elif rhythm_type == "Bradicardia":
            rr = np.random.normal(mean_rr, 0.04)

        elif rhythm_type == "Arritmia sinusal":
            rr = np.random.normal(mean_rr, 0.08)

        elif rhythm_type == "FA simulada":
            rr = np.random.normal(mean_rr, 0.14)

        elif rhythm_type == "Extrasistole":
            # cada algunos latidos aparece uno prematuro
            if len(rr_list) > 0 and len(rr_list) % 6 == 0:
                rr = np.random.normal(mean_rr * 0.65, 0.02)
            else:
                rr = np.random.normal(mean_rr, 0.04)
        else:
            rr = np.random.normal(mean_rr, 0.03)

        rr = np.clip(rr, 0.35, 2.0)
        rr_list.append(rr)
        total += rr

    return rr_list

def generate_ecg_signal(fs=1000, duration=10, hr=75, amplitude=1.0,
                        rhythm_type="Normal", noise=0.02,
                        baseline_wander=0.05, line_noise=0.01):
    """
    Genera ECG completo.
    """
    n = int(fs * duration)
    t = np.arange(n) / fs
    signal = np.zeros(n)

    rr_list = generate_rr_series(duration, hr, rhythm_type)

    beat_times = []
    current_time = 0.0

    for i, rr in enumerate(rr_list):
        if current_time >= duration:
            break

        local_type = rhythm_type
        if rhythm_type == "Extrasistole" and i > 0 and i % 6 == 0:
            local_type = "Extrasistole"
        elif rhythm_type == "Extrasistole":
            local_type = "Normal"

        tb, beat = single_beat_template(fs=fs, rr=rr, amplitude=amplitude, rhythm_type=local_type)

        start = int(current_time * fs)
        end = min(start + len(beat), n)
        beat_cut = beat[:max(0, end - start)]

        if start < n:
            signal[start:end] += beat_cut

            # R aproximado
            r_time = current_time + 0.40 * rr
            if r_time < duration:
                beat_times.append(r_time)

        current_time += rr

    # Deriva respiratoria
    signal += baseline_wander * np.sin(2 * np.pi * 0.25 * t)

    # Interferencia de red
    signal += line_noise * np.sin(2 * np.pi * 50 * t)

    # Ruido blanco
    signal += noise * np.random.randn(n)

    df = pd.DataFrame({
        "Time": t,
        "ECG": signal
    })

    return df, np.array(beat_times)

# ============================================================
# DETECCION SIMPLE DE PICOS R
# ============================================================

def detect_r_peaks(ecg, fs, threshold_scale=0.55, refractory_ms=250):
    x = ecg.copy()
    thr = np.mean(x) + threshold_scale * np.std(x)
    refractory = int(refractory_ms * fs / 1000)

    peaks = []
    last_peak = -refractory

    for i in range(1, len(x) - 1):
        if x[i] > thr and x[i] > x[i - 1] and x[i] > x[i + 1]:
            if i - last_peak >= refractory:
                peaks.append(i)
                last_peak = i

    return np.array(peaks)

# ============================================================
# FEATURES BIOMEDICAS
# ============================================================

def compute_ecg_features(df, fs):
    ecg = df["ECG"].values
    t = df["Time"].values

    peaks = detect_r_peaks(ecg, fs)

    if len(peaks) >= 2:
        rr = np.diff(peaks) / fs
        hr_inst = 60 / rr
        hr_mean = np.mean(hr_inst)
        rr_mean = np.mean(rr)
        rr_std = np.std(rr)
    else:
        rr = np.array([])
        hr_inst = np.array([])
        hr_mean = np.nan
        rr_mean = np.nan
        rr_std = np.nan

    rms = np.sqrt(np.mean(ecg**2))
    vmax = np.max(ecg)
    vmin = np.min(ecg)
    ptp = vmax - vmin

    freqs = np.fft.rfftfreq(len(ecg), d=1/fs)
    spec = np.abs(np.fft.rfft(ecg))**2
    spec_sum = np.sum(spec) + 1e-12
    fmean = np.sum(freqs * spec) / spec_sum
    cumsum = np.cumsum(spec)
    fmed = freqs[np.searchsorted(cumsum, cumsum[-1] / 2)]

    features = {
        "FC media [lpm]": hr_mean,
        "RR medio [s]": rr_mean,
        "SDNN aprox [s]": rr_std,
        "RMS": rms,
        "Vmax [mV]": vmax,
        "Vmin [mV]": vmin,
        "Pico a pico [mV]": ptp,
        "Fmean [Hz]": fmean,
        "Fmed [Hz]": fmed,
        "Latidos detectados": len(peaks)
    }

    return features, peaks, rr, hr_inst

def rhythm_quality_label(hr_mean, rr_std):
    if np.isnan(hr_mean):
        return "Señal insuficiente"
    if hr_mean > 100:
        return "Taquicardia"
    if hr_mean < 60:
        return "Bradicardia"
    if rr_std > 0.12:
        return "Ritmo irregular"
    return "Ritmo estable"

# ============================================================
# DASHBOARD
# ============================================================

def build_ecg_dashboard(df, fs, selected_rhythm):
    ecg = df["ECG"].values
    t = df["Time"].values

    features, peaks, rr, hr_inst = compute_ecg_features(df, fs)

    hr_mean = features["FC media [lpm]"]
    rr_std = features["SDNN aprox [s]"]
    quality = rhythm_quality_label(hr_mean, rr_std)

    # Espectro
    freqs = np.fft.rfftfreq(len(ecg), d=1/fs)
    mag = np.abs(np.fft.rfft(ecg))

    # ECG filtrado simple visual
    env = moving_average(np.abs(ecg), max(1, int(0.03 * fs)))

    fig = make_subplots(
        rows=4, cols=2,
        subplot_titles=(
            "Señal ECG",
            "Latidos detectados",
            "Envolvente / energía",
            "Espectro de frecuencia",
            "HR instantánea",
            "Intervalos RR",
            "Frecuencia cardíaca",
            "Estado del ritmo"
        ),
        specs=[
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "xy"}, {"type": "bar"}],
            [{"type": "indicator"}, {"type": "indicator"}]
        ],
        vertical_spacing=0.08,
        horizontal_spacing=0.10
    )

    # ECG
    fig.add_trace(
        go.Scatter(
            x=t, y=ecg,
            mode="lines",
            name="ECG",
            line=dict(width=1.5)
        ),
        row=1, col=1
    )

    # R peaks
    fig.add_trace(
        go.Scatter(
            x=t, y=ecg,
            mode="lines",
            name="ECG",
            line=dict(width=1.2),
            showlegend=False
        ),
        row=1, col=2
    )

    if len(peaks) > 0:
        fig.add_trace(
            go.Scatter(
                x=t[peaks],
                y=ecg[peaks],
                mode="markers",
                name="Picos R",
                marker=dict(size=8, symbol="circle")
            ),
            row=1, col=2
        )

    # Envolvente
    fig.add_trace(
        go.Scatter(
            x=t, y=env,
            mode="lines",
            name="Envolvente",
            line=dict(width=2),
            showlegend=False
        ),
        row=2, col=1
    )

    # FFT
    fig.add_trace(
        go.Scatter(
            x=freqs, y=mag,
            mode="lines",
            name="FFT",
            line=dict(width=1.5),
            showlegend=False
        ),
        row=2, col=2
    )

    # HR instantánea
    if len(hr_inst) > 0:
        hr_t = t[peaks][1:]
        fig.add_trace(
            go.Scatter(
                x=hr_t,
                y=hr_inst,
                mode="lines+markers",
                name="HR",
                showlegend=False
            ),
            row=3, col=1
        )

    # RR
    if len(rr) > 0:
        fig.add_trace(
            go.Bar(
                x=np.arange(1, len(rr) + 1),
                y=rr,
                text=[f"{v:.2f}" for v in rr],
                textposition="outside",
                showlegend=False
            ),
            row=3, col=2
        )

    # Gauge FC
    hr_value = 0 if np.isnan(hr_mean) else float(hr_mean)
    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=hr_value,
            title={"text": f"FC media<br><span style='font-size:12px'>{selected_rhythm}</span>"},
            gauge={
                "axis": {"range": [0, 180]},
                "bar": {"thickness": 0.3},
                "steps": [
                    {"range": [0, 60], "color": "#dbeafe"},
                    {"range": [60, 100], "color": "#d1fae5"},
                    {"range": [100, 180], "color": "#fee2e2"}
                ]
            }
        ),
        row=4, col=1
    )

    # Estado ritmo
    state_map = {
        "Ritmo estable": 85,
        "Ritmo irregular": 55,
        "Taquicardia": 95,
        "Bradicardia": 40,
        "Señal insuficiente": 15
    }
    state_value = state_map.get(quality, 50)

    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=state_value,
            title={"text": f"{quality}"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"thickness": 0.3},
                "steps": [
                    {"range": [0, 35], "color": "#dbeafe"},
                    {"range": [35, 70], "color": "#fef3c7"},
                    {"range": [70, 100], "color": "#d1fae5"}
                ]
            }
        ),
        row=4, col=2
    )

    fig.update_layout(
        title="Dashboard Interactivo de Señales Cardíacas (ECG)",
        height=1400,
        width=1500,
        template="plotly_white",
        margin=dict(l=40, r=40, t=80, b=40)
    )

    fig.update_xaxes(title_text="Tiempo [s]", row=1, col=1)
    fig.update_yaxes(title_text="Amplitud [mV]", row=1, col=1)

    fig.update_xaxes(title_text="Tiempo [s]", row=1, col=2)
    fig.update_yaxes(title_text="Amplitud [mV]", row=1, col=2)

    fig.update_xaxes(title_text="Tiempo [s]", row=2, col=1)
    fig.update_yaxes(title_text="Nivel", row=2, col=1)

    fig.update_xaxes(title_text="Frecuencia [Hz]", range=[0, 40], row=2, col=2)
    fig.update_yaxes(title_text="Magnitud", row=2, col=2)

    fig.update_xaxes(title_text="Tiempo [s]", row=3, col=1)
    fig.update_yaxes(title_text="lpm", row=3, col=1)

    fig.update_xaxes(title_text="N° intervalo", row=3, col=2)
    fig.update_yaxes(title_text="RR [s]", row=3, col=2)

    return fig, features

# ============================================================
# EXPORTACION
# ============================================================

LAST = {"df": None, "features": None, "fig": None}

def export_csv():
    if LAST["df"] is None:
        print("Primero generá una simulación.")
        return
    LAST["df"].to_csv("ecg_simulado.csv", index=False)
    pd.DataFrame([LAST["features"]]).to_csv("ecg_metricas.csv", index=False)
    print("Exportados: ecg_simulado.csv y ecg_metricas.csv")

def export_html():
    if LAST["fig"] is None:
        print("Primero generá una simulación.")
        return
    LAST["fig"].write_html("dashboard_ecg.html")
    print("Exportado: dashboard_ecg.html")

# ============================================================
# WIDGETS
# ============================================================

rhythm_widget = widgets.Dropdown(
    options=["Normal", "Taquicardia", "Bradicardia", "Arritmia sinusal", "Extrasistole", "FA simulada"],
    value="Normal",
    description="Ritmo:",
    layout=widgets.Layout(width="320px")
)

hr_widget = widgets.IntSlider(
    value=75, min=35, max=160, step=1,
    description="FC [lpm]:",
    layout=widgets.Layout(width="420px")
)

amp_widget = widgets.FloatSlider(
    value=1.0, min=0.4, max=2.0, step=0.05,
    description="Amplitud:",
    readout_format=".2f",
    layout=widgets.Layout(width="420px")
)

noise_widget = widgets.FloatSlider(
    value=0.02, min=0.0, max=0.25, step=0.01,
    description="Ruido:",
    readout_format=".2f",
    layout=widgets.Layout(width="420px")
)

baseline_widget = widgets.FloatSlider(
    value=0.05, min=0.0, max=0.30, step=0.01,
    description="Deriva base:",
    readout_format=".2f",
    layout=widgets.Layout(width="420px")
)

line_widget = widgets.FloatSlider(
    value=0.01, min=0.0, max=0.10, step=0.005,
    description="Red 50 Hz:",
    readout_format=".3f",
    layout=widgets.Layout(width="420px")
)

duration_widget = widgets.FloatSlider(
    value=10.0, min=5.0, max=30.0, step=1.0,
    description="Duración [s]:",
    readout_format=".1f",
    layout=widgets.Layout(width="420px")
)

fs_widget = widgets.IntSlider(
    value=1000, min=250, max=4000, step=250,
    description="Fs [Hz]:",
    layout=widgets.Layout(width="420px")
)

btn_csv = widgets.Button(
    description="Exportar CSV",
    button_style="success",
    layout=widgets.Layout(width="180px")
)

btn_html = widgets.Button(
    description="Exportar HTML",
    button_style="info",
    layout=widgets.Layout(width="180px")
)

output = widgets.Output()

# ============================================================
# REFRESH
# ============================================================

def refresh_dashboard(*args):
    with output:
        clear_output(wait=True)

        fs = fs_widget.value
        duration = duration_widget.value
        hr = hr_widget.value
        rhythm = rhythm_widget.value
        amp = amp_widget.value
        noise = noise_widget.value
        baseline = baseline_widget.value
        line = line_widget.value

        df, beats = generate_ecg_signal(
            fs=fs,
            duration=duration,
            hr=hr,
            amplitude=amp,
            rhythm_type=rhythm,
            noise=noise,
            baseline_wander=baseline,
            line_noise=line
        )

        fig, features = build_ecg_dashboard(df, fs, rhythm)

        LAST["df"] = df
        LAST["features"] = features
        LAST["fig"] = fig

        fig.show()

        metricas = pd.DataFrame([features]).T.reset_index()
        metricas.columns = ["Métrica", "Valor"]

        print("Resumen biomédico ECG")
        print(f"Ritmo seleccionado: {rhythm}")
        print("")

        display(metricas.style.format({"Valor": "{:.4f}"}))

def on_csv_clicked(b):
    with output:
        export_csv()

def on_html_clicked(b):
    with output:
        export_html()

btn_csv.on_click(on_csv_clicked)
btn_html.on_click(on_html_clicked)

for w in [
    rhythm_widget, hr_widget, amp_widget, noise_widget,
    baseline_widget, line_widget, duration_widget, fs_widget
]:
    w.observe(refresh_dashboard, names="value")

# ============================================================
# INTERFAZ
# ============================================================

left_controls = widgets.VBox([
    rhythm_widget,
    hr_widget,
    amp_widget,
    noise_widget
])

right_controls = widgets.VBox([
    baseline_widget,
    line_widget,
    duration_widget,
    fs_widget,
    widgets.HBox([btn_csv, btn_html])
])

ui = widgets.HBox([left_controls, right_controls])

display(HTML("<h2>Simulador Interactivo de Señales Cardíacas (ECG)</h2>"))
display(HTML("<p>Modelo sintético orientado a docencia, análisis biomédico y prototipado en bioinstrumentación.</p>"))
display(ui)
display(output)

refresh_dashboard()

Output()

In [4]:
!pip install plotly ipywidgets pandas numpy

from google.colab import output
output.enable_custom_widget_manager()

import plotly.io as pio
pio.renderers.default = "colab"

In [5]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

np.random.seed(7)

# -----------------------------
# MODELO ECG
# -----------------------------
def gaussian(t, mu, sigma, amp):
    return amp * np.exp(-0.5 * ((t - mu) / sigma) ** 2)

def single_beat_template(fs=500, rr=0.8, amplitude=1.0, rhythm_type="Normal"):
    beat_len = int(rr * fs)
    t = np.linspace(0, rr, beat_len, endpoint=False)
    x = np.zeros_like(t)

    p_pos, q_pos, r_pos, s_pos, t_pos = 0.18*rr, 0.36*rr, 0.40*rr, 0.44*rr, 0.68*rr
    p_w, q_w, r_w, s_w, t_w = 0.035*rr, 0.010*rr, 0.012*rr, 0.012*rr, 0.08*rr
    p_a, q_a, r_a, s_a, t_a = 0.12*amplitude, -0.15*amplitude, 1.0*amplitude, -0.25*amplitude, 0.35*amplitude

    if rhythm_type == "Taquicardia":
        t_a = 0.28 * amplitude
    elif rhythm_type == "Bradicardia":
        t_a = 0.40 * amplitude
    elif rhythm_type == "FA simulada":
        p_a = 0.0
    elif rhythm_type == "Extrasistole":
        q_a, r_a, s_a, t_a = -0.22*amplitude, 1.15*amplitude, -0.35*amplitude, 0.2*amplitude

    x += gaussian(t, p_pos, p_w, p_a)
    x += gaussian(t, q_pos, q_w, q_a)
    x += gaussian(t, r_pos, r_w, r_a)
    x += gaussian(t, s_pos, s_w, s_a)
    x += gaussian(t, t_pos, t_w, t_a)
    return t, x

def generate_rr_series(duration, hr, rhythm_type):
    mean_rr = 60.0 / hr
    rr_list, total = [], 0
    while total < duration + 1:
        if rhythm_type == "Normal":
            rr = np.random.normal(mean_rr, 0.03)
        elif rhythm_type == "Taquicardia":
            rr = np.random.normal(mean_rr, 0.02)
        elif rhythm_type == "Bradicardia":
            rr = np.random.normal(mean_rr, 0.04)
        elif rhythm_type == "Arritmia sinusal":
            rr = np.random.normal(mean_rr, 0.08)
        elif rhythm_type == "FA simulada":
            rr = np.random.normal(mean_rr, 0.14)
        elif rhythm_type == "Extrasistole":
            rr = np.random.normal(mean_rr * 0.65, 0.02) if (len(rr_list) > 0 and len(rr_list) % 6 == 0) else np.random.normal(mean_rr, 0.04)
        else:
            rr = np.random.normal(mean_rr, 0.03)

        rr = np.clip(rr, 0.35, 2.0)
        rr_list.append(rr)
        total += rr
    return rr_list

def generate_ecg_signal(fs=500, duration=10, hr=75, amplitude=1.0,
                        rhythm_type="Normal", noise=0.02,
                        baseline_wander=0.03, line_noise=0.01):
    n = int(fs * duration)
    t = np.arange(n) / fs
    signal = np.zeros(n)
    rr_list = generate_rr_series(duration, hr, rhythm_type)

    beat_times = []
    current_time = 0.0

    for i, rr in enumerate(rr_list):
        if current_time >= duration:
            break

        local_type = rhythm_type
        if rhythm_type == "Extrasistole":
            local_type = "Extrasistole" if (i > 0 and i % 6 == 0) else "Normal"

        tb, beat = single_beat_template(fs=fs, rr=rr, amplitude=amplitude, rhythm_type=local_type)
        start = int(current_time * fs)
        end = min(start + len(beat), n)
        if start < n:
            signal[start:end] += beat[:max(0, end-start)]
            r_time = current_time + 0.40 * rr
            if r_time < duration:
                beat_times.append(r_time)
        current_time += rr

    signal += baseline_wander * np.sin(2 * np.pi * 0.25 * t)
    signal += line_noise * np.sin(2 * np.pi * 50 * t)
    signal += noise * np.random.randn(n)

    return pd.DataFrame({"Time": t, "ECG": signal}), np.array(beat_times)

def detect_r_peaks(ecg, fs, threshold_scale=0.55, refractory_ms=250):
    thr = np.mean(ecg) + threshold_scale * np.std(ecg)
    refractory = int(refractory_ms * fs / 1000)
    peaks, last_peak = [], -refractory
    for i in range(1, len(ecg)-1):
        if ecg[i] > thr and ecg[i] > ecg[i-1] and ecg[i] > ecg[i+1]:
            if i - last_peak >= refractory:
                peaks.append(i)
                last_peak = i
    return np.array(peaks)

def compute_metrics(df, fs):
    ecg = df["ECG"].values
    t = df["Time"].values
    peaks = detect_r_peaks(ecg, fs)

    if len(peaks) >= 2:
        rr = np.diff(peaks) / fs
        hr_inst = 60 / rr
        hr_mean = float(np.mean(hr_inst))
        rr_std = float(np.std(rr))
        spo2_fake = np.clip(99 - rr_std*30 - max(0, hr_mean-100)*0.03, 92, 100)
    else:
        rr = np.array([])
        hr_inst = np.array([])
        hr_mean = 0
        rr_std = 0
        spo2_fake = 98

    if hr_mean > 100:
        state = "TAQUI"
    elif hr_mean < 60 and hr_mean > 0:
        state = "BRADI"
    elif rr_std > 0.12:
        state = "IRREG"
    else:
        state = "NORMAL"

    return {
        "peaks": peaks,
        "rr": rr,
        "hr_inst": hr_inst,
        "hr_mean": hr_mean,
        "rr_std": rr_std,
        "spo2_fake": spo2_fake,
        "state": state
    }

def build_icu_monitor(df, fs, rhythm_label):
    ecg = df["ECG"].values
    t = df["Time"].values
    m = compute_metrics(df, fs)

    alarm = m["state"] in ["TAQUI", "BRADI", "IRREG"]
    alarm_text = "ALARMA" if alarm else "SIN ALARMA"

    fig = make_subplots(
        rows=2, cols=2,
        row_heights=[0.72, 0.28],
        column_widths=[0.72, 0.28],
        specs=[
            [{"type": "xy"}, {"type": "indicator"}],
            [{"type": "indicator"}, {"type": "indicator"}]
        ],
        subplot_titles=("ECG Monitor", "Frecuencia cardíaca", "SpO₂ estimada", "Estado")
    )

    fig.add_trace(
        go.Scatter(
            x=t, y=ecg,
            mode="lines",
            line=dict(color="#00FF66", width=2),
            name="ECG"
        ),
        row=1, col=1
    )

    if len(m["peaks"]) > 0:
        fig.add_trace(
            go.Scatter(
                x=t[m["peaks"]],
                y=ecg[m["peaks"]],
                mode="markers",
                marker=dict(size=7, color="#FFFFFF"),
                name="R"
            ),
            row=1, col=1
        )

    fig.add_trace(
        go.Indicator(
            mode="number+gauge",
            value=m["hr_mean"],
            title={"text": "HR bpm"},
            gauge={
                "axis": {"range": [0, 180]},
                "bar": {"color": "#00FF66"},
                "bgcolor": "#111111",
                "borderwidth": 1,
                "bordercolor": "#555555",
                "steps": [
                    {"range": [0, 60], "color": "#1f3b4d"},
                    {"range": [60, 100], "color": "#123524"},
                    {"range": [100, 180], "color": "#4a1111"},
                ]
            },
            number={"font": {"color": "#00FF66", "size": 40}}
        ),
        row=1, col=2
    )

    fig.add_trace(
        go.Indicator(
            mode="number+gauge",
            value=m["spo2_fake"],
            title={"text": "SpO₂ %"},
            gauge={
                "axis": {"range": [80, 100]},
                "bar": {"color": "#00D4FF"},
                "bgcolor": "#111111",
                "steps": [
                    {"range": [80, 92], "color": "#4a1111"},
                    {"range": [92, 95], "color": "#4d431f"},
                    {"range": [95, 100], "color": "#123524"},
                ]
            },
            number={"font": {"color": "#00D4FF", "size": 34}}
        ),
        row=2, col=1
    )

    fig.add_trace(
        go.Indicator(
            mode="number",
            value=1 if alarm else 0,
            title={"text": f"{m['state']}<br>{alarm_text}<br><span style='font-size:12px'>{rhythm_label}</span>"},
            number={"font": {"color": "#FF4D4D" if alarm else "#00FF66", "size": 42}}
        ),
        row=2, col=2
    )

    fig.update_layout(
        title="Monitor UTI - ECG Bedside",
        template="plotly_dark",
        height=800,
        width=1350,
        paper_bgcolor="#000000",
        plot_bgcolor="#000000",
        font=dict(color="#EAEAEA"),
        margin=dict(l=30, r=30, t=70, b=30),
        showlegend=False
    )

    fig.update_xaxes(
        title_text="Tiempo [s]",
        showgrid=True,
        gridcolor="#133313",
        zeroline=False,
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="mV",
        showgrid=True,
        gridcolor="#133313",
        zeroline=False,
        row=1, col=1
    )

    return fig

# -----------------------------
# WIDGETS
# -----------------------------
rhythm_widget = widgets.Dropdown(
    options=["Normal", "Taquicardia", "Bradicardia", "Arritmia sinusal", "Extrasistole", "FA simulada"],
    value="Normal",
    description="Ritmo:"
)
hr_widget = widgets.IntSlider(value=78, min=35, max=160, description="FC:")
amp_widget = widgets.FloatSlider(value=1.0, min=0.4, max=2.0, step=0.05, description="Amp:")
noise_widget = widgets.FloatSlider(value=0.02, min=0.0, max=0.20, step=0.01, description="Ruido:")
duration_widget = widgets.FloatSlider(value=12, min=5, max=30, step=1, description="Duración:")
fs_widget = widgets.IntSlider(value=500, min=250, max=2000, step=250, description="Fs:")
output_a = widgets.Output()

def refresh_a(*args):
    with output_a:
        clear_output(wait=True)
        df, _ = generate_ecg_signal(
            fs=fs_widget.value,
            duration=duration_widget.value,
            hr=hr_widget.value,
            amplitude=amp_widget.value,
            rhythm_type=rhythm_widget.value,
            noise=noise_widget.value
        )
        fig = build_icu_monitor(df, fs_widget.value, rhythm_widget.value)
        fig.show()

for w in [rhythm_widget, hr_widget, amp_widget, noise_widget, duration_widget, fs_widget]:
    w.observe(refresh_a, names="value")

display(HTML("<h2>Versión A - Monitor UTI ECG</h2>"))
display(widgets.HBox([
    widgets.VBox([rhythm_widget, hr_widget, amp_widget]),
    widgets.VBox([noise_widget, duration_widget, fs_widget])
]))
display(output_a)
refresh_a()

Output()

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

np.random.seed(12)

# -----------------------------
# REUTILIZO MODELO ECG
# -----------------------------
def gaussian(t, mu, sigma, amp):
    return amp * np.exp(-0.5 * ((t - mu) / sigma) ** 2)

def single_beat_template(fs=1000, rr=1.0, amplitude=1.0, rhythm_type="Normal"):
    beat_len = int(rr * fs)
    t = np.linspace(0, rr, beat_len, endpoint=False)
    x = np.zeros_like(t)

    p_pos, q_pos, r_pos, s_pos, t_pos = 0.18*rr, 0.36*rr, 0.40*rr, 0.44*rr, 0.68*rr
    p_w, q_w, r_w, s_w, t_w = 0.035*rr, 0.010*rr, 0.012*rr, 0.012*rr, 0.080*rr
    p_a, q_a, r_a, s_a, t_a = 0.12*amplitude, -0.15*amplitude, 1.00*amplitude, -0.25*amplitude, 0.35*amplitude

    if rhythm_type == "Taquicardia":
        t_a = 0.28 * amplitude
    elif rhythm_type == "Bradicardia":
        t_a = 0.40 * amplitude
    elif rhythm_type == "FA simulada":
        p_a = 0.0
    elif rhythm_type == "Extrasistole":
        q_a, r_a, s_a, t_a = -0.20*amplitude, 1.15*amplitude, -0.35*amplitude, 0.22*amplitude

    x += gaussian(t, p_pos, p_w, p_a)
    x += gaussian(t, q_pos, q_w, q_a)
    x += gaussian(t, r_pos, r_w, r_a)
    x += gaussian(t, s_pos, s_w, s_a)
    x += gaussian(t, t_pos, t_w, t_a)
    return t, x

def generate_rr_series(duration, hr, rhythm_type):
    mean_rr = 60.0 / hr
    rr_list, total = [], 0
    while total < duration + 1:
        if rhythm_type == "Normal":
            rr = np.random.normal(mean_rr, 0.03)
        elif rhythm_type == "Taquicardia":
            rr = np.random.normal(mean_rr, 0.02)
        elif rhythm_type == "Bradicardia":
            rr = np.random.normal(mean_rr, 0.04)
        elif rhythm_type == "Arritmia sinusal":
            rr = np.random.normal(mean_rr, 0.08)
        elif rhythm_type == "FA simulada":
            rr = np.random.normal(mean_rr, 0.14)
        elif rhythm_type == "Extrasistole":
            rr = np.random.normal(mean_rr * 0.65, 0.02) if (len(rr_list) > 0 and len(rr_list) % 6 == 0) else np.random.normal(mean_rr, 0.04)
        else:
            rr = np.random.normal(mean_rr, 0.03)
        rr = np.clip(rr, 0.35, 2.0)
        rr_list.append(rr)
        total += rr
    return rr_list

def generate_ecg_signal(fs=1000, duration=10, hr=75, amplitude=1.0,
                        rhythm_type="Normal", noise=0.02,
                        baseline_wander=0.05, line_noise=0.01):
    n = int(fs * duration)
    t = np.arange(n) / fs
    signal = np.zeros(n)
    rr_list = generate_rr_series(duration, hr, rhythm_type)

    beat_times = []
    current_time = 0.0

    for i, rr in enumerate(rr_list):
        if current_time >= duration:
            break

        local_type = rhythm_type
        if rhythm_type == "Extrasistole":
            local_type = "Extrasistole" if (i > 0 and i % 6 == 0) else "Normal"

        tb, beat = single_beat_template(fs=fs, rr=rr, amplitude=amplitude, rhythm_type=local_type)
        start = int(current_time * fs)
        end = min(start + len(beat), n)

        if start < n:
            signal[start:end] += beat[:max(0, end-start)]
            r_time = current_time + 0.40 * rr
            if r_time < duration:
                beat_times.append(r_time)

        current_time += rr

    signal += baseline_wander * np.sin(2 * np.pi * 0.25 * t)
    signal += line_noise * np.sin(2 * np.pi * 50 * t)
    signal += noise * np.random.randn(n)

    return pd.DataFrame({"Time": t, "ECG": signal}), np.array(beat_times)

def detect_r_peaks(ecg, fs, threshold_scale=0.55, refractory_ms=250):
    thr = np.mean(ecg) + threshold_scale * np.std(ecg)
    refractory = int(refractory_ms * fs / 1000)
    peaks, last_peak = [], -refractory
    for i in range(1, len(ecg)-1):
        if ecg[i] > thr and ecg[i] > ecg[i-1] and ecg[i] > ecg[i+1]:
            if i - last_peak >= refractory:
                peaks.append(i)
                last_peak = i
    return np.array(peaks)

def compute_features(df, fs):
    ecg = df["ECG"].values
    peaks = detect_r_peaks(ecg, fs)

    if len(peaks) >= 2:
        rr = np.diff(peaks) / fs
        hr_inst = 60 / rr
        hr_mean = float(np.mean(hr_inst))
        rr_mean = float(np.mean(rr))
        rr_std = float(np.std(rr))
    else:
        rr = np.array([])
        hr_inst = np.array([])
        hr_mean = np.nan
        rr_mean = np.nan
        rr_std = np.nan

    rms = float(np.sqrt(np.mean(ecg**2)))
    vmax = float(np.max(ecg))
    vmin = float(np.min(ecg))
    ptp = vmax - vmin
    freqs = np.fft.rfftfreq(len(ecg), d=1/fs)
    spec = np.abs(np.fft.rfft(ecg))**2
    spec_sum = np.sum(spec) + 1e-12
    fmean = float(np.sum(freqs * spec) / spec_sum)
    cumsum = np.cumsum(spec)
    fmed = float(freqs[np.searchsorted(cumsum, cumsum[-1]/2)])

    return {
        "peaks": peaks,
        "rr": rr,
        "hr_inst": hr_inst,
        "FC media": hr_mean,
        "RR medio": rr_mean,
        "SDNN": rr_std,
        "RMS": rms,
        "Pico a pico": ptp,
        "Fmed": fmed,
        "Fmean": fmean
    }

def semaforo_fc(fc):
    if np.isnan(fc):
        return "Sin dato"
    if fc < 60:
        return "Bradicardia"
    if fc <= 100:
        return "Normal"
    return "Taquicardia"

def semaforo_rr(sdnn):
    if np.isnan(sdnn):
        return "Sin dato"
    if sdnn < 0.05:
        return "Baja variabilidad"
    if sdnn < 0.12:
        return "Variabilidad normal"
    return "Alta irregularidad"

def build_powerbi_dashboard(df, fs, rhythm):
    t = df["Time"].values
    ecg = df["ECG"].values
    feat = compute_features(df, fs)

    freqs = np.fft.rfftfreq(len(ecg), d=1/fs)
    mag = np.abs(np.fft.rfft(ecg))

    fig = make_subplots(
        rows=4, cols=3,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"colspan": 2, "type": "xy"}, None, {"type": "xy"}],
            [{"type": "xy"}, {"type": "bar"}, {"type": "bar"}],
            [{"colspan": 3, "type": "table"}, None, None]
        ],
        subplot_titles=(
            "FC media", "Variabilidad RR", "Estado clínico",
            "ECG temporal", "Espectro",
            "HR instantánea", "Intervalos RR", "Indicadores",
            "Resumen"
        ),
        vertical_spacing=0.09,
        horizontal_spacing=0.08
    )

    fc_val = 0 if np.isnan(feat["FC media"]) else feat["FC media"]
    sdnn_ms = 0 if np.isnan(feat["SDNN"]) else feat["SDNN"] * 1000

    fig.add_trace(
        go.Indicator(
            mode="number+delta",
            value=fc_val,
            delta={"reference": 75},
            title={"text": "bpm"},
            number={"suffix": " lpm"}
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Indicator(
            mode="number+delta",
            value=sdnn_ms,
            delta={"reference": 50},
            title={"text": "SDNN"},
            number={"suffix": " ms"}
        ),
        row=1, col=2
    )

    estado_num = 90 if semaforo_fc(feat["FC media"]) == "Normal" else 55
    if semaforo_fc(feat["FC media"]) in ["Bradicardia", "Taquicardia"]:
        estado_num = 35

    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=estado_num,
            title={"text": f"{semaforo_fc(feat['FC media'])}<br><span style='font-size:12px'>{rhythm}</span>"},
            gauge={
                "axis": {"range": [0, 100]},
                "bar": {"thickness": 0.32},
                "steps": [
                    {"range": [0, 40], "color": "#f8d7da"},
                    {"range": [40, 70], "color": "#fff3cd"},
                    {"range": [70, 100], "color": "#d1e7dd"},
                ]
            }
        ),
        row=1, col=3
    )

    fig.add_trace(
        go.Scatter(x=t, y=ecg, mode="lines", name="ECG", line=dict(width=1.8)),
        row=2, col=1
    )
    if len(feat["peaks"]) > 0:
        fig.add_trace(
            go.Scatter(
                x=t[feat["peaks"]],
                y=ecg[feat["peaks"]],
                mode="markers",
                name="Picos R",
                marker=dict(size=7)
            ),
            row=2, col=1
        )

    fig.add_trace(
        go.Scatter(x=freqs, y=mag, mode="lines", name="FFT", line=dict(width=1.6), showlegend=False),
        row=2, col=3
    )

    if len(feat["hr_inst"]) > 0:
        fig.add_trace(
            go.Scatter(
                x=t[feat["peaks"]][1:],
                y=feat["hr_inst"],
                mode="lines+markers",
                name="HR instantánea",
                showlegend=False
            ),
            row=3, col=1
        )

    if len(feat["rr"]) > 0:
        fig.add_trace(
            go.Bar(
                x=np.arange(1, len(feat["rr"]) + 1),
                y=feat["rr"],
                text=[f"{v:.2f}" for v in feat["rr"]],
                textposition="outside",
                showlegend=False
            ),
            row=3, col=2
        )

    labels = ["RMS", "Pico-Pico", "Fmed", "Fmean"]
    values = [feat["RMS"], feat["Pico a pico"], feat["Fmed"], feat["Fmean"]]
    fig.add_trace(
        go.Bar(
            x=labels,
            y=values,
            text=[f"{v:.2f}" for v in values],
            textposition="outside",
            showlegend=False
        ),
        row=3, col=3
    )

    resumen = pd.DataFrame({
        "Métrica": ["FC media", "RR medio", "SDNN", "RMS", "Pico a pico", "Fmed", "Fmean", "Clasificación FC", "Clasificación RR"],
        "Valor": [
            f"{feat['FC media']:.2f} lpm" if not np.isnan(feat["FC media"]) else "N/A",
            f"{feat['RR medio']:.3f} s" if not np.isnan(feat["RR medio"]) else "N/A",
            f"{feat['SDNN']*1000:.1f} ms" if not np.isnan(feat["SDNN"]) else "N/A",
            f"{feat['RMS']:.3f}",
            f"{feat['Pico a pico']:.3f} mV",
            f"{feat['Fmed']:.2f} Hz",
            f"{feat['Fmean']:.2f} Hz",
            semaforo_fc(feat["FC media"]),
            semaforo_rr(feat["SDNN"]),
        ]
    })

    fig.add_trace(
        go.Table(
            header=dict(values=list(resumen.columns)),
            cells=dict(values=[resumen["Métrica"], resumen["Valor"]])
        ),
        row=4, col=1
    )

    fig.update_layout(
        title="Dashboard Premium ECG - Estilo Power BI Biomédico",
        template="plotly_white",
        height=1200,
        width=1550,
        margin=dict(l=30, r=30, t=80, b=30)
    )

    fig.update_xaxes(title_text="Tiempo [s]", row=2, col=1)
    fig.update_yaxes(title_text="mV", row=2, col=1)
    fig.update_xaxes(title_text="Frecuencia [Hz]", range=[0, 40], row=2, col=3)
    fig.update_yaxes(title_text="Magnitud", row=2, col=3)
    fig.update_xaxes(title_text="Tiempo [s]", row=3, col=1)
    fig.update_yaxes(title_text="lpm", row=3, col=1)
    fig.update_xaxes(title_text="N° intervalo", row=3, col=2)
    fig.update_yaxes(title_text="RR [s]", row=3, col=2)
    fig.update_xaxes(title_text="Indicador", row=3, col=3)

    return fig

# -----------------------------
# WIDGETS
# -----------------------------
rhythm_b = widgets.Dropdown(
    options=["Normal", "Taquicardia", "Bradicardia", "Arritmia sinusal", "Extrasistole", "FA simulada"],
    value="Normal",
    description="Ritmo:"
)
hr_b = widgets.IntSlider(value=75, min=35, max=160, description="FC:")
amp_b = widgets.FloatSlider(value=1.0, min=0.4, max=2.0, step=0.05, description="Amp:")
noise_b = widgets.FloatSlider(value=0.02, min=0.0, max=0.25, step=0.01, description="Ruido:")
baseline_b = widgets.FloatSlider(value=0.05, min=0.0, max=0.30, step=0.01, description="Deriva:")
line_b = widgets.FloatSlider(value=0.01, min=0.0, max=0.10, step=0.005, description="50Hz:")
duration_b = widgets.FloatSlider(value=12, min=5, max=30, step=1, description="Duración:")
fs_b = widgets.IntSlider(value=1000, min=250, max=4000, step=250, description="Fs:")
output_b = widgets.Output()

def refresh_b(*args):
    with output_b:
        clear_output(wait=True)
        df, _ = generate_ecg_signal(
            fs=fs_b.value,
            duration=duration_b.value,
            hr=hr_b.value,
            amplitude=amp_b.value,
            rhythm_type=rhythm_b.value,
            noise=noise_b.value,
            baseline_wander=baseline_b.value,
            line_noise=line_b.value
        )
        fig = build_powerbi_dashboard(df, fs_b.value, rhythm_b.value)
        fig.show()

for w in [rhythm_b, hr_b, amp_b, noise_b, baseline_b, line_b, duration_b, fs_b]:
    w.observe(refresh_b, names="value")

display(HTML("<h2>Versión B - Dashboard Premium ECG</h2>"))
display(widgets.HBox([
    widgets.VBox([rhythm_b, hr_b, amp_b, noise_b]),
    widgets.VBox([baseline_b, line_b, duration_b, fs_b])
]))
display(output_b)
refresh_b()

Output()